**Mounting google drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Import Package**

In [2]:
import io
import pandas as pd

**Install Package and Obtain Processing Time**

In [3]:
!pip install ipython-autotime

In [4]:
%load_ext autotime

time: 285 µs (started: 2024-06-05 14:13:03 +00:00)


**Check Files in Colab Notebooks Folder in gdrive**

In [5]:
!ls '/content/drive/My Drive/Colab Notebooks/dataset'


data_iris.csv  pima_indians_diabetes.csv
time: 106 ms (started: 2024-06-05 14:13:05 +00:00)


**Read data in pandas and Display the first few lines of data .csv**

In [6]:
sample_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/dataset/data_iris.csv', header=0)

time: 27.9 ms (started: 2024-06-05 14:13:16 +00:00)


In [7]:
sample_df.head()

,F1,F2,F3,F4,Label
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


time: 49.6 ms (started: 2024-06-05 14:13:18 +00:00)


**Import Package for Pytorch to design ANN**

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd

time: 1.75 s (started: 2024-06-05 14:13:21 +00:00)


**Define the input, X and label as Y**

In [9]:
# Load the dataset
file_path = '/content/drive/MyDrive/Colab Notebooks/dataset/data_iris.csv'
data = pd.read_csv(file_path)
label_encoder = LabelEncoder()
data['Encoded_Label'] = label_encoder.fit_transform(data['Label'])

# Convert to numpy arrays
X = data[['F1', 'F2', 'F3', 'F4']].values.astype(float)
Y = label_encoder.transform(data['Label'])  # Direct integer encoding

# One-hot encoding of the labels using PyTorch
Y = torch.nn.functional.one_hot(torch.tensor(Y), num_classes=len(set(Y)))

# Displaying the result
print(X[:5])
print(Y[:5])  # Showing the first five one-hot encoded labels


[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]
tensor([[1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0]])
time: 20 ms (started: 2024-06-05 14:13:25 +00:00)


**Splitting The Data for Train and Test**

In [10]:
from torch.utils.data import DataLoader, TensorDataset

# Splitting the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Convert numpy arrays to torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)


time: 3.79 ms (started: 2024-06-05 14:13:28 +00:00)


<ipython-input-10-f4ca31b4283e>:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype=torch.float32)
<ipython-input-10-f4ca31b4283e>:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_test = torch.tensor(y_test, dtype=torch.float32)


**Create Class for dataset and dataloader**

DataLoader is a fundamental part of the data loading utility that makes it easier to iterate over batches of data.



In [11]:
# Create Tensor datasets
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

# Load data into DataLoader
train_loader = DataLoader(dataset=train_dataset, batch_size=10, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=10, shuffle=False)

time: 847 µs (started: 2024-06-05 14:13:31 +00:00)


**Create ANN Model**

In [12]:
# Define the model
class NeuralNet(nn.Module):
    def __init__(self):
        super(NeuralNet, self).__init__()
        self.layer1 = nn.Linear(4, 10)  # 4 input features, 10 neurons in hidden layer
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(10, 3)  # 3 output classes

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

model = NeuralNet()

time: 3.18 ms (started: 2024-06-05 14:13:35 +00:00)


**Define Optimizer and Loss Function for Training Phase**

In [13]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

time: 979 ms (started: 2024-06-05 14:13:39 +00:00)


**Train the Network**

In [14]:
# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    for inputs, labels in train_loader:
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, torch.max(labels, 1)[1])

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Calculate training accuracy
    with torch.no_grad():
        correct = 0
        total = 0
        for inputs, labels in train_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == torch.max(labels, 1)[1]).sum().item()
        training_accuracy = 100 * correct / total

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Training Accuracy: {training_accuracy:.2f}%')


Epoch [10/100], Loss: 0.2686, Training Accuracy: 95.83%
Epoch [20/100], Loss: 0.2004, Training Accuracy: 98.33%
Epoch [30/100], Loss: 0.1549, Training Accuracy: 98.33%
Epoch [40/100], Loss: 0.0543, Training Accuracy: 97.50%
Epoch [50/100], Loss: 0.0457, Training Accuracy: 97.50%
Epoch [60/100], Loss: 0.0191, Training Accuracy: 97.50%
Epoch [70/100], Loss: 0.0494, Training Accuracy: 97.50%
Epoch [80/100], Loss: 0.1805, Training Accuracy: 98.33%
Epoch [90/100], Loss: 0.0566, Training Accuracy: 97.50%
Epoch [100/100], Loss: 0.0337, Training Accuracy: 97.50%
time: 1.72 s (started: 2024-06-05 14:13:42 +00:00)


**Test the Model**

In [15]:
# Test the model
model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    correct = 0
    total = 0
    for inputs, labels in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == torch.max(labels, 1)[1]).sum().item()

    print(f'Accuracy of the network on the test data: {100 * correct / total} %')

Accuracy of the network on the test data: 100.0 %
time: 4.13 ms (started: 2024-06-05 14:13:47 +00:00)


**Testing on One Specific Test Data**

In [16]:
# Testing a specific sample from the test set
sample_index = 2  # Change this index to test a different sample
sample_data = X_test[sample_index]
sample_label = y_test[sample_index]

# Prediction
sample_prediction = model(sample_data)
predicted_class = torch.argmax(sample_prediction).item()
actual_class = torch.argmax(sample_label).item()
print(f'Sample Data: {sample_data}')
print(f'Predicted Class: {predicted_class}, Actual Class: {actual_class}')

Sample Data: tensor([7.7000, 2.6000, 6.9000, 2.3000])
Predicted Class: 2, Actual Class: 2
time: 4.38 ms (started: 2024-06-05 14:13:53 +00:00)


## Export for deployment — two files

Run the training and evaluation cells above, then run this cell. It generates
`deployment/iris/model.pt` (trained weights) and `deployment/iris/model.json`
(architecture, feature order, labels, preprocessing, and an example).

This notebook uses raw `F1`, `F2`, `F3`, and `F4` values without scaling.
Class labels are exported in the exact order learned by `LabelEncoder`.
The export verifies that saved weights preserve the notebook's predictions.

Copy **both files** to `models/iris/` in the deployment repository. Add `"iris"`
to the `models` list in `config/ann.json`, restart both the API and worker, and
reload the ANN Inference tab. PIMA remains the default unless you change
`default_model`. Replace both files together whenever you retrain.


In [ ]:
# EXPORT FOR DEPLOYMENT

# Run after training and evaluation. No manual JSON editing is needed.
import copy
import io
import json
from pathlib import Path

MODEL_ID = "iris"
DISPLAY_NAME = "Iris ANN"

# The notebook uses raw numeric features, with no fitted scaler.
deployment_model = nn.Sequential(
    copy.deepcopy(model.layer1), nn.ReLU(), copy.deepcopy(model.layer2),
).cpu().eval()
check_input = X_test[:1].detach().cpu()
feature_names = ["F1", "F2", "F3", "F4"]
input_config = {
    "type": "tabular", "dtype": "float32", "shape": [None, 4],
    "features": feature_names, "preprocessing": {"type": "none"},
}
output_config = {
    "task": "multiclass_classification", "shape": [None, 3], "target": "Label",
    # Preserve LabelEncoder's output-index order; do not manually reorder labels.
    "labels": [str(label) for label in label_encoder.classes_],
    "postprocessing": {"type": "softmax", "dim": 1},
}
example = {"features": dict(zip(feature_names, check_input[0].tolist()))}

# Describe the exact layer order so serving does not need this notebook's class.
architecture = []
for layer in deployment_model:
    if isinstance(layer, nn.Linear):
        architecture.append({
            "type": "linear", "in_features": layer.in_features,
            "out_features": layer.out_features, "bias": layer.bias is not None,
        })
    elif isinstance(layer, nn.ReLU):
        architecture.append({"type": "relu"})
    elif isinstance(layer, nn.Flatten):
        architecture.append({
            "type": "flatten", "start_dim": layer.start_dim, "end_dim": layer.end_dim,
        })
    else:
        raise ValueError(f"Export does not support layer: {type(layer).__name__}")

config = {
    "schema_version": 1,
    "model_id": MODEL_ID,
    "display_name": DISPLAY_NAME,
    "model_version": "1",
    "framework": "pytorch",
    "pytorch_version": str(torch.__version__),
    "weights": {"file": "model.pt", "format": "state_dict"},
    "architecture": {"type": "sequential", "layers": architecture},
    "input": input_config,
    "output": output_config,
    "example": example,
}

# Check that the portable model and saved weights preserve the original outputs.
# Moving copies to CPU leaves the trained model on its original device.
original_cpu = copy.deepcopy(model).cpu().eval()
deployment_model.eval()
weights_buffer = io.BytesIO()
torch.save(deployment_model.state_dict(), weights_buffer)
weights_buffer.seek(0)
reloaded_model = copy.deepcopy(deployment_model)
reloaded_model.load_state_dict(torch.load(weights_buffer, map_location="cpu", weights_only=True))
with torch.inference_mode():
    expected = original_cpu(check_input)
    actual = reloaded_model(check_input)
torch.testing.assert_close(actual, expected)

# The folder contains the two files students copy together for deployment.
export_dir = Path("deployment") / MODEL_ID
export_dir.mkdir(parents=True, exist_ok=True)
config_text = json.dumps(config, indent=2, allow_nan=False) + "\n"
(export_dir / "model.pt").write_bytes(weights_buffer.getvalue())
(export_dir / "model.json").write_text(config_text, encoding="utf-8")

print("Export verified: predictions match the trained model.")
print("Export folder:", export_dir.resolve())
print("Copy BOTH model.pt and model.json into models/" + MODEL_ID + "/.")
print("Keep these files together; export both again after retraining.")
